# Trabalho Prático Final - Parte 2
## Interpretação do Melhor Modelo

Objetivo: interpretar o melhor modelo treinado usando permutation importance e, opcionalmente, SHAP.

### Setup Inicial
Esta célula importa as bibliotecas necessárias para a análise, configura um `RANDOM_SEED` para reprodutibilidade e cria os diretórios de saída para figuras e métricas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings

from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs("../results/figures/interpretability", exist_ok=True)
os.makedirs("../results/metrics", exist_ok=True)

### Carregamento dos Dados de Teste
Esta célula carrega o conjunto de dados de teste (`t2_test.csv`) e separa as features (`X_test`) da variável alvo (`y_test`). Também imprime as dimensões dos dataframes resultantes.

In [ ]:
target_col = "Overall"

test_df = pd.read_csv("../data/processed/t2_test.csv")
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(X_test.shape)
print(y_test.shape)

### Identificação do Melhor Modelo
Esta célula lê o nome do melhor modelo treinado de um arquivo de texto, que será usado posteriormente para carregar o modelo.

In [ ]:
with open("../results/metrics/best_model_name.txt", "r", encoding="utf-8") as f:
    best_model_name = f.read().strip()

print(f"Melhor modelo: {best_model_name}")

### Carregamento do Melhor Modelo
Esta célula constrói o caminho para o arquivo do melhor modelo (`.pkl`) e o carrega usando `joblib.load`.

In [ ]:
model_filename = best_model_name.lower().replace(" ", "_") + "_best_model.pkl"
model_path = f"../results/models/{model_filename}"

best_model = joblib.load(model_path)

print(f"Modelo carregado de: {model_path}")

### Cálculo da Permutation Importance
Esta célula calcula a Permutation Importance para o melhor modelo nos dados de teste. Ela utiliza o F1-score como métrica de pontuação e repete o processo 10 vezes para obter uma média e desvio padrão da importância de cada feature. Os resultados são salvos em um arquivo CSV e exibidos.

In [ ]:
perm_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
})

importance_df = importance_df.sort_values("importance_mean", ascending=False)

importance_df.to_csv("../results/metrics/t2_permutation_importance.csv", index=False)

importance_df.head(20)

### Visualização da Permutation Importance
Esta célula gera um gráfico de barras mostrando as 20 principais features com base em sua Permutation Importance (queda média no F1-score). As barras de erro representam o desvio padrão da importância. O gráfico é salvo como uma imagem PNG.

In [ ]:
top_features = importance_df.head(20)

plt.figure(figsize=(10, 8))

# Plotar as barras sem as barras de erro
sns.barplot(
    data=top_features,
    x="importance_mean",
    y="feature"
)

# Adicionar as barras de erro manualmente
# A posição y para as barras é 0, 1, 2, ... para cada feature
y_positions = np.arange(len(top_features))
plt.errorbar(
    x=top_features["importance_mean"],
    y=y_positions,
    xerr=top_features["importance_std"],
    fmt='none', # Não desenhar linha entre os pontos de erro
    c='black',  # Cor das barras de erro
    capsize=4   # Tamanho das "tampas" das barras de erro
)

plt.title(f"Top 20 Features por Permutation Importance - {best_model_name}")
plt.xlabel("Queda média no F1-score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("../results/figures/interpretability/permutation_importance_top20.png", dpi=300)
plt.show()

### Análise de Interpretabilidade SHAP (Opcional)
Esta célula tenta realizar uma análise de interpretabilidade usando a biblioteca SHAP. Ela amostra os dados de teste para reduzir o custo computacional, extrai o estimador final do modelo (se for um pipeline) e gera um `summary_plot` SHAP. Se houver algum erro (por exemplo, SHAP não instalado ou incompatibilidade do modelo), ele informará que a análise SHAP não foi executada.

In [ ]:
try:
    import shap

    print("SHAP disponível. Gerando análise SHAP...")

    # Amostra para reduzir custo computacional
    X_sample = X_test.sample(
        n=min(500, len(X_test)),
        random_state=RANDOM_SEED
    )

    # Para modelos em pipeline, tentar extrair modelo final e dados transformados
    preprocessed_sample = best_model[:-1].transform(X_sample)
    final_estimator = best_model[-1]

    explainer = shap.TreeExplainer(final_estimator)
    shap_values = explainer.shap_values(preprocessed_sample)

    shap.summary_plot(
        shap_values,
        preprocessed_sample,
        feature_names=X_test.columns,
        show=False
    )

    plt.tight_layout()
    plt.savefig("../results/figures/interpretability/shap_summary_plot.png", dpi=300)
    plt.show()

except Exception as e:
    print("SHAP não foi executado.")
    print(f"Motivo: {e}")
    print("A análise principal de interpretabilidade será baseada em permutation importance.")

## Texto-base para o relatório

A interpretação do modelo final foi realizada por meio de permutation importance. Esse método avalia a relevância de cada atributo medindo a queda no desempenho do modelo quando os valores desse atributo são embaralhados. Assim, atributos cuja permutação reduz mais fortemente o F1-score são considerados mais importantes para a decisão do modelo.

Foram analisadas as 20 features mais relevantes. Os resultados permitem identificar quais descritores moleculares tiveram maior impacto na predição de mutagenicidade Ames. Essa análise oferece uma visão global do comportamento do modelo, embora não estabeleça causalidade entre os descritores e a mutagenicidade.